In [0]:
/Volumes/daily_assignments/assignments/day3

In [0]:
data = [
    (1, "A", 5000),
    (2, "B", 6000),
    (3, "C", 7000)
]

columns = ["id", "name", "salary"]

df = spark.createDataFrame(data, columns)
df.display()

id,name,salary
1,A,5000
2,B,6000
3,C,7000


### Write as Delta Table

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .save("/Volumes/daily_assignments/assignments/day3")

### Read Delta Table

In [0]:
df = spark.read.format("delta").load("/Volumes/daily_assignments/assignments/day3")
df.display()

id,name,salary
1,A,5000
2,B,6000
3,C,7000


# Insert

In [0]:
new_data = [(4, "D", 8000)]
new_df = spark.createDataFrame(new_data, columns)

new_df.write.format("delta")\
    .mode("append")\
    .save("/Volumes/daily_assignments/assignments/day3")

# Update

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/Volumes/daily_assignments/assignments/day3")

delta_table.update(
    condition = "id = 2",
    set = {"salary": "6500"}
)

DataFrame[num_affected_rows: bigint]

# Delete

In [0]:
delta_table.delete("id = 1")

DataFrame[num_affected_rows: bigint]

### MERGE INTO

### How do we handle incremental load with updates?”

Existing data

In [0]:
target_df = spark.read.format("delta").load("/Volumes/daily_assignments/assignments/day3")
target_df.display()

id,name,salary
2,B,6500
3,C,7000
4,D,8000


New data

In [0]:
updates = [
    (2, "B", 7000),  # update
    (5, "E", 9000)   # insert
]

updates_df = spark.createDataFrame(updates, columns)
updates_df.display()

id,name,salary
2,B,7000
5,E,9000


Merge

In [0]:
delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set={
    "salary": "source.salary"
}).whenNotMatchedInsert(values={
    "id": "source.id",
    "name": "source.name",
    "salary": "source.salary"
}).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### If id exists → UPDATE
### If id not exists → INSERT
### This is UPSERT

### Schema Evolution

Problem:
New column arrives

In [0]:
new_data = [(6, "F", 10000, "India")]
new_columns = ["id", "name", "salary", "country"]

df_new = spark.createDataFrame(new_data, new_columns)
df_new.display()

id,name,salary,country
6,F,10000,India


### Enable schema evolution:

In [0]:
df_new.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/Volumes/daily_assignments/assignments/day3")

In [0]:
df_read=spark.read.format('delta').load("/Volumes/daily_assignments/assignments/day3")
df_read.display()


id,name,salary,country
6,F,10000,India
3,C,7000,null
4,D,8000,null
2,B,7000,null
5,E,9000,null


### Time Travel

### View History

In [0]:
delta_table.history().display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2026-04-16T06:49:03.000Z,72574808668492,pagadalapraveena2004@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(980272541629529),748803c2-13ce-4080-8876-09dd3e6f99ec,0416-055413-rtdlqcp5-v2n,7,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1199)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
7,2026-04-16T06:44:45.000Z,72574808668492,pagadalapraveena2004@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(980272541629529),d896b0c0-9d55-4d37-95a2-e3e6649bef9d,0416-055413-rtdlqcp5-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3053, p25FileSize -> 1083, numDeletionVectorsRemoved -> 1, minFileSize -> 1083, numAddedFiles -> 1, maxFileSize -> 1083, p75FileSize -> 1083, p50FileSize -> 1083, numAddedBytes -> 1083)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
6,2026-04-16T06:44:43.000Z,72574808668492,pagadalapraveena2004@gmail.com,MERGE,"Map(predicate -> [""(id#13745L = id#13748L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(980272541629529),d896b0c0-9d55-4d37-95a2-e3e6649bef9d,0416-055413-rtdlqcp5-v2n,5,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 2008, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3937, materializeSourceTimeMs -> 220, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1461, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2132)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
5,2026-04-16T06:42:17.000Z,72574808668492,pagadalapraveena2004@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(980272541629529),4cd423e4-d500-4cf2-9b44-11c218a686cf,0416-055413-rtdlqcp5-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1061, p25FileSize -> 1045, numDeletionVectorsRemoved -> 1, minFileSize -> 1045, numAddedFiles -> 1, maxFileSize -> 1045, p75FileSize -> 1045, p50FileSize -> 1045, numAddedBytes -> 1045)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
4,2026-04-16T06:42:15.000Z,72574808668492,pagadalapraveena2004@gmail.com,DELETE,"Map(predicate -> [""(id#13432L = 1)""])",null,List(980272541629529),4cd423e4-d500-4cf2-9b44-11c218a686cf,0416-055413-rtdlqcp5-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1434, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1069, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 364)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
3,2026-04-16T06:41:15.000Z,72574808668492,pagadalapraveena2004@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(980272541629529),6efcfcd6-a86c-40a4-90e7-b558f8c321e3,0416-055413-rtdlqcp5-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3051, p25FileSize -> 1061, numDeletionVectorsRemoved -> 1, minFileSize -> 1061, numAddedFiles -> 1, maxFileSize -> 1061, p75FileSize -> 1061, p50FileSize -> 1061, numAddedBytes -> 1061)",null,Databricks-

### Read Old Version

In [0]:
df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/daily_assignments/assignments/day3")
df_old.display()

id,name,salary
1,A,5000
2,B,6000
3,C,7000


### Restore Table

In [0]:
delta_table.restoreToVersion(0)

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]